In [1]:
import requests  #get dta from website
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

In [132]:
def scrape_movie(movie):

    url = movie["movie_url"]

    # Request movie page
    try:
        response = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0",
                "Accept-Language": "en-US,en;q=0.9"
            },
            timeout=20
        )

    except requests.exceptions.RequestException as e:
        print("Connection error:", movie["title"])
        return None

    # Stop immediately if TMDB throttles us
    if response.status_code == 429:
        raise RuntimeError("TMDB_THROTTLED")

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # JSON-LD data
    scripts = soup.find_all(
        "script",
        type="application/ld+json"
    )

    if not scripts:
        return None

    text = scripts[0].text.strip()
    text = text.replace("/* <![CDATA[ */", "")
    text = text.replace("/* ]]> */", "")

    data = json.loads(text)

    # Status, language, budget, revenue
    status = None
    language = None
    budget = None
    revenue = None

    for p in soup.find_all("p"):

        text = p.get_text(" ", strip=True)

        if text.startswith("Status"):
            status = text.replace("Status", "").strip()

        elif text.startswith("Original Language"):
            language = text.replace(
                "Original Language", ""
            ).strip()

        elif text.startswith("Budget"):
            budget = text.replace("Budget", "").strip()

        elif text.startswith("Revenue"):
            revenue = text.replace("Revenue", "").strip()

    # Keywords
    keywords = []

    keyword_section = soup.find(
        "section",
        class_="keywords"
    )

    if keyword_section:
        for a in keyword_section.find_all("a"):
            keywords.append(
                a.get_text(strip=True)
            )

    # Cast
    cast = []

    cast_section = soup.find(
        "section",
        class_="top_billed"
    )

    if cast_section:
        for card in cast_section.select("li.card"):

            actor = card.find("p")
            character = card.find(
                "p",
                class_="character"
            )

            if actor:
                cast.append({
                    "actor": actor.get_text(strip=True),
                    "character": character.get_text(
                        strip=True
                    ) if character else None
                })

    # Director and writers
    director = None
    writers = []

    for person in soup.select("li.profile"):

        name = person.find("a")
        role = person.find(
            "p",
            class_="character"
        )

        if not name or not role:
            continue

        name = name.get_text(strip=True)
        role = role.get_text(strip=True)

        if "Director" in role:
            director = name

        if "Writer" in role:
            writers.append(name)

    # Content Score
    content_score = None

    content_section = soup.find(
        "section",
        class_="content_score"
    )

    if content_section:
        score = content_section.select_one(
            ".content_score_wrapper > .content_score p"
        )

        if score:
            content_score = score.get_text(strip=True)

    # Final movie record
    movie_data = {
        "movie_id": movie["movie_id"],
        "title": data.get("name"),
        "movie_url": movie["movie_url"],
        "poster_url": movie["poster_url"],
        "release_date": movie["release_date"],
        "genres": data.get("genre", []),
        "overview": data.get("description"),
        "user_score": data.get(
            "aggregateRating", {}
        ).get("ratingValue"),
        "runtime": data.get("duration"),
        "director": director,
        "writers": writers,
        "cast": cast,
        "status": status,
        "original_language": language,
        "budget": budget,
        "revenue": revenue,
        "keywords": keywords,
        "content_score": content_score
    }

    return movie_data

In [31]:
def get_movie_list(page=1):

    url = f"https://www.themoviedb.org/movie?page={page}"

    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept-Language": "en-US,en;q=0.9"
        }
    )

    if response.status_code != 200:
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    movies = []

    for card in soup.select("div[data-object-id]"):

        link = card.find("a", href=lambda x: x and "/movie/" in x)

        if link:

            href = link["href"]
            title = card.find("h2")
            date = card.find("span", class_="release_date")
            poster = card.find("img")

            movies.append({
                "movie_id": href.split("/movie/")[1].split("-")[0],
                "title": title.get_text(strip=True) if title else None,
                "movie_url": "https://www.themoviedb.org" + href,
                "poster_url": poster.get("src") if poster else None,
                "release_date": date.get_text(strip=True) if date else None
            })

    return movies

In [137]:
def get_all_movies():

    all_movies = []
    page = 1

    while True:

        print("Scraping page:", page)

        page_movies = get_movie_list(page)

        if len(page_movies) == 0:
            print("No more movies found.")
            break

        all_movies.extend(page_movies)

        print("Movies collected:", len(all_movies))

        page += 1

    return all_movies

In [64]:
all_movies = get_all_movies()

print("Total movies collected:", len(all_movies))

Scraping page: 1
Movies collected: 20
Scraping page: 2
Movies collected: 40
Scraping page: 3
Movies collected: 60
Scraping page: 4
Movies collected: 80
Scraping page: 5
Movies collected: 100
Scraping page: 6
Movies collected: 120
Scraping page: 7
Movies collected: 140
Scraping page: 8
Movies collected: 160
Scraping page: 9
Movies collected: 180
Scraping page: 10
Movies collected: 200
Scraping page: 11
Movies collected: 220
Scraping page: 12
Movies collected: 240
Scraping page: 13
Movies collected: 260
Scraping page: 14
Movies collected: 280
Scraping page: 15
Movies collected: 300
Scraping page: 16
Movies collected: 320
Scraping page: 17
Movies collected: 340
Scraping page: 18
Movies collected: 360
Scraping page: 19
Movies collected: 380
Scraping page: 20
Movies collected: 400
Scraping page: 21
Movies collected: 420
Scraping page: 22
Movies collected: 440
Scraping page: 23
Movies collected: 460
Scraping page: 24
Movies collected: 480
Scraping page: 25
Movies collected: 500
Scraping page

In [65]:
unique_movies = {}

for movie in all_movies:
    unique_movies[movie["movie_id"]] = movie

all_movies = list(unique_movies.values())

print("Before deduplication:", 10000)
print("After deduplication:", len(all_movies))

Before deduplication: 10000
After deduplication: 10000


In [122]:
def scrape_reviews(movie, max_reviews=3):

    url = movie["movie_url"] + "/reviews"

    try:
        response = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0",
                "Accept-Language": "en-US,en;q=0.9"
            },
            timeout=20
        )

    except requests.exceptions.RequestException as e:
        print("Connection error:", movie["title"])
        return []

    if response.status_code != 200:
        print("Failed:", response.status_code, movie["title"])
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    reviews = []

    for card in soup.select("div.card"):

        review_link = card.find(
            "a",
            href=lambda x: x and "/review/" in x
        )

        if not review_link:
            continue

        review_id = review_link["href"].split("/review/")[1].split("?")[0]

        author = None
        author_tag = card.select_one("h5 a")

        if author_tag:
            author = author_tag.get_text(strip=True)

        review_date = None
        date_tag = card.select_one("h5")

        if date_tag:
            date_text = date_tag.get_text(" ", strip=True)

            if " on " in date_text:
                review_date = date_text.split(" on ", 1)[1].strip()

        review_text = None
        teaser = card.select_one(".teaser")

        if teaser:
            review_text = teaser.get_text(" ", strip=True)

        reviews.append({
            "movie_id": movie["movie_id"],
            "review_id": review_id,
            "author": author,
            "review_date": review_date,
            "review_text": review_text
        })

        if len(reviews) >= max_reviews:
            break

    return reviews

In [124]:
len(reviews_data)

8900

In [130]:
all_movies_df = pd.DataFrame(all_movies)

all_movies_df.to_csv(
    "movie_list_10000.csv",
    index=False
)

print("Movies saved:", len(all_movies_df))

Movies saved: 10000


In [135]:
import time
import pandas as pd

movies_data_final = []
cast_data_final = []

for i, movie in enumerate(all_movies, start=1):

    try:
        result = scrape_movie(movie)

    except RuntimeError as e:

        if str(e) == "TMDB_THROTTLED":

            # Save everything collected so far
            pd.DataFrame(movies_data_final).to_csv(
                "movies_checkpoint.csv",
                index=False
            )

            pd.DataFrame(cast_data_final).to_csv(
                "movie_cast_checkpoint.csv",
                index=False
            )

            print()
            print("TMDB THROTTLED THE CONNECTION.")
            print("Scrape stopped at movie:", i)
            print("Movies collected:", len(movies_data_final))
            break

        raise

    if result is not None:

        cast = result.pop("cast", [])

        movies_data_final.append(result)

        for person in cast:

            cast_data_final.append({
                "movie_id": movie["movie_id"],
                "actor": person["actor"],
                "character": person["character"]
            })

    if i % 50 == 0:

        pd.DataFrame(movies_data_final).to_csv(
            "movies_checkpoint.csv",
            index=False
        )

        pd.DataFrame(cast_data_final).to_csv(
            "movie_cast_checkpoint.csv",
            index=False
        )

        print(
            "Movies processed:", i,
            "| Movies collected:", len(movies_data_final),
            "| Cast records:", len(cast_data_final)
        )

    time.sleep(2)

Connection error: The Mongoose
Connection error: Ice Cream Man
Connection error: Avengers: Infinity War
Movies processed: 50 | Movies collected: 47 | Cast records: 410
Movies processed: 100 | Movies collected: 97 | Cast records: 850
Movies processed: 150 | Movies collected: 147 | Cast records: 1291
Movies processed: 200 | Movies collected: 197 | Cast records: 1733
Movies processed: 250 | Movies collected: 247 | Cast records: 2181
Movies processed: 300 | Movies collected: 297 | Cast records: 2622
Movies processed: 350 | Movies collected: 347 | Cast records: 3065
Connection error: Gladiator II
Movies processed: 400 | Movies collected: 396 | Cast records: 3495
Movies processed: 450 | Movies collected: 446 | Cast records: 3934
Movies processed: 500 | Movies collected: 496 | Cast records: 4376
Movies processed: 550 | Movies collected: 546 | Cast records: 4824
Movies processed: 600 | Movies collected: 596 | Cast records: 5264
Movies processed: 650 | Movies collected: 646 | Cast records: 5710

In [136]:
movies = pd.DataFrame(movies_data_final)
movie_cast = pd.DataFrame(cast_data_final)

# Save detailed movie data
movies.to_csv(
    "movies.csv",
    index=False
)

# Save cast data
movie_cast.to_csv(
    "movie_cast.csv",
    index=False
)

print("Movies saved:", len(movies))
print("Cast records saved:", len(movie_cast))

Movies saved: 6764
Cast records saved: 59675


In [138]:
movies = pd.DataFrame(movies_data_final)
movie_cast = pd.DataFrame(cast_data_final)

movies.to_csv("movies.csv", index=False)
movie_cast.to_csv("movie_cast.csv", index=False)

In [2]:
import pandas as pd
import numpy as np
import ast
import re

In [3]:
movies = pd.read_csv("movies.csv")
movie_cast = pd.read_csv("movie_cast.csv")
movie_reviews = pd.read_csv("movie_reviews.csv")

In [4]:
movie_cast = pd.read_csv("movie_cast_checkpoint.csv")

In [5]:
movies.head()

,movie_id,title,movie_url,poster_url,release_date,genres,overview,user_score,runtime,director,writers,status,original_language,budget,revenue,keywords,content_score
0,969681,Spider-Man: Brand New Day,https://www.themoviedb.org/movie/969681-spider...,https://media.themoviedb.org/t/p/w220_and_h330...,"July 30, 2026","['Science Fiction', 'Action', 'Adventure']",Fighting crime full-time as Spider-Man in a wo...,7.860,PT2H25M,Destin Daniel Cretton,"['Erik Sommers', 'Chris McKenna']",Released,English,"$225,000,000","$2,478,502,286","['new york city', 'secret identity', 'hero', '...",100
1,1423191,Resident Evil,https://www.themoviedb.org/movie/1423191-resid...,https://media.themoviedb.org/t/p/w220_and_h330...,"September 17, 2026","['Horror', 'Science Fiction', 'Adventure']",Medical courier Bryan unwittingly finds himsel...,7.299,PT1H35M,Zach Cregger,"['Zach Cregger', 'Shay Hatten']",Released,English,"$75,000,000","$123,281,812","['outbreak', 'survival', 'zombie', 'based on v...",100
2,1101383,The End of Oak Street,https://www.themoviedb.org/movie/1101383-the-e...,https://media.themoviedb.org/t/p/w220_and_h330...,"August 14, 2026","['Science Fiction', 'Mystery', 'Thriller']",After a mysterious cosmic event rips Oak Stree...,7.003,PT1H40M,David Robert Mitchell,['David Robert Mitchell'],Released,English,"$80,000,000","$120,363,694","['teleportation', 'giant snake', 'family relat...",100
3,1204680,Coyote vs. Acme,https://www.themoviedb.org/movie/1204680-coyot...,https://media.themoviedb.org/t/p/w220_and_h330...,"August 20, 2026","['Comedy', 'Adventure', 'Family']",After Acme products fail him one too many time...,7.556,PT1H43M,Dave Green,[],Released,English,"$72,000,000","$79,825,108","['judge', 'lawsuit', 'lawyer', 'courtroom', 'u...",100
4,1368337,The Odyssey,https://www.themoviedb.org/movie/1368337-the-o...,https://media.themoviedb.org/t/p/w220_and_h330...,"July 17, 2026","['Adventure', 'Action', 'Fantasy']","Odysseus, the legendary King of Ithaca, embark...",8.000,PT2H53M,Christopher Nolan,['Christopher Nolan'],Released,English,"$250,000,000","$1,723,918,419","['greek mythology', 'trojan war', 'ship', 'his...",100


In [6]:
movies_clean = movies.copy()
movie_cast_clean = movie_cast.copy()
movie_reviews_clean = movie_reviews.copy()

In [7]:
movies.shape

(6764, 17)

In [8]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 6764 entries, 0 to 6763
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           6764 non-null   int64  
 1   title              6764 non-null   str    
 2   movie_url          6764 non-null   str    
 3   poster_url         6758 non-null   str    
 4   release_date       6764 non-null   str    
 5   genres             6764 non-null   str    
 6   overview           6743 non-null   str    
 7   user_score         6639 non-null   float64
 8   runtime            6701 non-null   str    
 9   director           6749 non-null   str    
 10  writers            6764 non-null   str    
 11  status             6764 non-null   str    
 12  original_language  6764 non-null   str    
 13  budget             6764 non-null   str    
 14  revenue            6764 non-null   str    
 15  keywords           6764 non-null   str    
 16  content_score      6764 non-null   

In [9]:
movies.isna().sum()

movie_id               0
title                  0
movie_url              0
poster_url             6
release_date           0
genres                 0
overview              21
user_score           125
runtime               63
director              15
writers                0
status                 0
original_language      0
budget                 0
revenue                0
keywords               0
content_score          0
dtype: int64

In [10]:
(movies == "").sum()

movie_id             0
title                0
movie_url            0
poster_url           0
release_date         0
genres               0
overview             0
user_score           0
runtime              0
director             0
writers              0
status               0
original_language    0
budget               0
revenue              0
keywords             0
content_score        0
dtype: int64

In [14]:
movies_clean["release_date"] = pd.to_datetime(
    movies_clean["release_date"],
    errors="coerce"
)

movies_clean["release_year"] = movies_clean["release_date"].dt.year

In [15]:
movies_clean[["release_date", "release_year"]].head()

,release_date,release_year
0,2026-07-30,2026
1,2026-09-17,2026
2,2026-08-14,2026
3,2026-08-20,2026
4,2026-07-17,2026


In [16]:
movies_clean[
    ["user_score", "content_score", "runtime", "budget", "revenue"]
].head(10)

,user_score,content_score,runtime,budget,revenue
0,7.860,100,PT2H25M,"$225,000,000","$2,478,502,286"
1,7.299,100,PT1H35M,"$75,000,000","$123,281,812"
2,7.003,100,PT1H40M,"$80,000,000","$120,363,694"
3,7.556,100,PT1H43M,"$72,000,000","$79,825,108"
4,8.000,100,PT2H53M,"$250,000,000","$1,723,918,419"
5,8.073,100,PT2H3M,-,-
6,6.723,100,PT1H36M,"$40,000,000","$20,824,283"
7,8.330,100,PT1H50M,-,-
8,7.600,100,PT1H35M,-,-
9,7.393,100,PT1H55M,"$250,000,000","$308,159,595"


In [17]:
movies_clean[
    ["user_score", "content_score", "runtime", "budget", "revenue"]
].dtypes

user_score       float64
content_score      int64
runtime              str
budget               str
revenue              str
dtype: object

In [18]:
def runtime_to_minutes(runtime):
    if pd.isna(runtime):
        return np.nan

    hours = 0
    minutes = 0

    hour_match = re.search(r"(\d+)H", runtime)
    minute_match = re.search(r"(\d+)M", runtime)

    if hour_match:
        hours = int(hour_match.group(1))

    if minute_match:
        minutes = int(minute_match.group(1))

    return hours * 60 + minutes


movies_clean["runtime_minutes"] = movies_clean["runtime"].apply(
    runtime_to_minutes
)

In [19]:
movies_clean[["runtime", "runtime_minutes"]].head(10)

,runtime,runtime_minutes
0,PT2H25M,145.0
1,PT1H35M,95.0
2,PT1H40M,100.0
3,PT1H43M,103.0
4,PT2H53M,173.0
5,PT2H3M,123.0
6,PT1H36M,96.0
7,PT1H50M,110.0
8,PT1H35M,95.0
9,PT1H55M,115.0


In [20]:
movies_clean

,movie_id,title,movie_url,poster_url,release_date,genres,overview,user_score,runtime,director,writers,status,original_language,budget,revenue,keywords,content_score,release_year,runtime_minutes
0,969681,Spider-Man: Brand New Day,https://www.themoviedb.org/movie/969681-spider...,https://media.themoviedb.org/t/p/w220_and_h330...,2026-07-30,"['Science Fiction', 'Action', 'Adventure']",Fighting crime full-time as Spider-Man in a wo...,7.860,PT2H25M,Destin Daniel Cretton,"['Erik Sommers', 'Chris McKenna']",Released,English,"$225,000,000","$2,478,502,286","['new york city', 'secret identity', 'hero', '...",100,2026,145.0
1,1423191,Resident Evil,https://www.themoviedb.org/movie/1423191-resid...,https://media.themoviedb.org/t/p/w220_and_h330...,2026-09-17,"['Horror', 'Science Fiction', 'Adventure']",Medical courier Bryan unwittingly finds himsel...,7.299,PT1H35M,Zach Cregger,"['Zach Cregger', 'Shay Hatten']",Released,English,"$75,000,000","$123,281,812","['outbreak', 'survival', 'zombie', 'based on v...",100,2026,95.0
2,1101383,The End of Oak Street,https://www.themoviedb.org/movie/1101383-the-e...,https://media.themoviedb.org/t/p/w220_and_h330...,2026-08-14,"['Science Fiction', 'Mystery', 'Thriller']",After a mysterious cosmic event rips Oak Stree...,7.003,PT1H40M,David Robert Mitchell,['David Robert Mitchell'],Released,English,"$80,000,000","$120,363,694","['teleportation', 'giant snake', 'family relat...",100,2026,100.0
3,1204680,Coyote vs. Acme,https://www.themoviedb.org/movie/1204680-coyot...,https://media.themoviedb.org/t/p/w220_and_h330...,2026-08-20,"['Comedy', 'Adventure', 'Family']",After Acme products fail him one too many time...,7.556,PT1H43M,Dave Green,[],Released,English,"$72,000,000","$79,825,108","['judge', 'lawsuit', 'lawyer', 'courtroom', 'u...",100,2026,103.0
4,1368337,The Odyssey,https://www.themoviedb.org/movie/1368337-the-o...,https://media.themoviedb.org/t/p/w220_and_h330...,2026-07-17,"['Adventure', 'Action', 'Fantasy']","Odysseus, the legendary King of Ithaca, embark...",8.000,PT2H53M,Christopher Nolan,['Christopher Nolan'],Released,English,"$250,000,000","$1,723,918,419","['greek mythology', 'trojan war', 'ship', 'his...",100,2026,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6759,38363,Fair Game,https://www.themoviedb.org/movie/38363-fair-game,https://media.themoviedb.org/t/p/w220_and_h330...,2010-05-20,"['Drama', 'Thriller']",A devoted wife and mother leads a secret life ...,6.452,PT1H48M,Doug Liman,[],Released,English,"$22,000,000","$25,806,953","['central intelligence agency (cia)', 'nuclear...",100,2010,108.0
6760,16873,Battle for Terra,https://www.themoviedb.org/movie/16873-battle-...,https://media.themoviedb.org/t/p/w220_and_h330...,2007-09-06,"['Action', 'Adventure', 'Animation', 'Science ...","A peaceful alien planet faces annihilation, as...",6.129,PT1H25M,Aristomenis Tsirbas,[],Released,English,"$4,000,000","$6,101,046","['mutiny', 'cave', 'chase', 'invasion', 'surre...",100,2007,85.0
6761,11860,Sabrina,https://www.themoviedb.org/movie/11860-sabrina,https://media.themoviedb.org/t/p/w220_and_h330...,1995-12-15,"['Romance', 'Drama', 'Comedy']","After her return from school in Paris, a playb...",6.206,PT2H7M,Sydney Pollack,[],Released,English,"$58,000,000","$53,672,080","['paris, france', 'sibling relationship', 'cha...",100,1995,127.0
6762,9642,Asterix and the Vikings,https://www.themoviedb.org/movie/9642-asterix-...,https://media.themoviedb.org/t/p/w220_and_h330...,2006-04-12,"['Family', 'Animation', 'Comedy', 'Adventure']",Asterix and Obelix have been given a tough mis...,6.105,PT1H18M,Jesper Møller,[],Released,French,-,"$22,501,596","['france', 'magic', 'village', 'vikings (norse...",100,2006,78.0


In [21]:
movies_clean["budget"] = (
    movies_clean["budget"]
    .replace("-", np.nan)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

movies_clean["revenue"] = (
    movies_clean["revenue"]
    .replace("-", np.nan)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [22]:
movies_clean[["budget", "revenue"]].head(10)

,budget,revenue
0,225000000.0,2.478502e+09
1,75000000.0,1.232818e+08
2,80000000.0,1.203637e+08
3,72000000.0,7.982511e+07
4,250000000.0,1.723918e+09
5,NaN,NaN
6,40000000.0,2.082428e+07
7,NaN,NaN
8,NaN,NaN
9,250000000.0,3.081596e+08


In [23]:
movies_clean[["genres", "writers", "keywords"]].head(10)

,genres,writers,keywords
0,"['Science Fiction', 'Action', 'Adventure']","['Erik Sommers', 'Chris McKenna']","['new york city', 'secret identity', 'hero', '..."
1,"['Horror', 'Science Fiction', 'Adventure']","['Zach Cregger', 'Shay Hatten']","['outbreak', 'survival', 'zombie', 'based on v..."
2,"['Science Fiction', 'Mystery', 'Thriller']",['David Robert Mitchell'],"['teleportation', 'giant snake', 'family relat..."
3,"['Comedy', 'Adventure', 'Family']",[],"['judge', 'lawsuit', 'lawyer', 'courtroom', 'u..."
4,"['Adventure', 'Action', 'Fantasy']",['Christopher Nolan'],"['greek mythology', 'trojan war', 'ship', 'his..."
5,"['Action', 'Horror', 'Science Fiction']","['Yeon Sang-ho', 'Choi Gyu-seok']","['zombie', 'vaccine', 'ambivalent']"
6,"['Action', 'Thriller']","['J.P. Davis', 'Lindsay Michel']","['revenge', 'conspiracy', 'one man army', 'car..."
7,"['Romance', 'Comedy']",[],"['based on novel or book', 'professor', 'love'..."
8,"['Action', 'Thriller']",['Tom Boyle'],[]
9,"['Family', 'Fantasy', 'Comedy', 'Adventure']",[],"['ocean', 'sailboat', 'sea', ""leaving one's fa..."


In [24]:
movies_clean[["genres", "writers", "keywords"]].dtypes

genres      str
writers     str
keywords    str
dtype: object

In [25]:
import ast

list_columns = ["genres", "writers", "keywords"]

for col in list_columns:
    movies_clean[col] = movies_clean[col].apply(
        lambda x: ast.literal_eval(x) if pd.notna(x) else []
    )

In [26]:
movies_clean[["genres", "writers", "keywords"]].head()

,genres,writers,keywords
0,"[Science Fiction, Action, Adventure]","[Erik Sommers, Chris McKenna]","[new york city, secret identity, hero, mutatio..."
1,"[Horror, Science Fiction, Adventure]","[Zach Cregger, Shay Hatten]","[outbreak, survival, zombie, based on video ga..."
2,"[Science Fiction, Mystery, Thriller]",[David Robert Mitchell],"[teleportation, giant snake, family relationsh..."
3,"[Comedy, Adventure, Family]",[],"[judge, lawsuit, lawyer, courtroom, unlikely f..."
4,"[Adventure, Action, Fantasy]",[Christopher Nolan],"[greek mythology, trojan war, ship, historical..."


In [27]:
movie_genres = movies_clean[["movie_id", "genres"]].explode("genres")

movie_genres = movie_genres.rename(
    columns={"genres": "genre"}
)

movie_genres = movie_genres[
    movie_genres["genre"].notna() &
    (movie_genres["genre"] != "")
]

movie_genres.head()

,movie_id,genre
0,969681,Science Fiction
0,969681,Action
0,969681,Adventure
1,1423191,Horror
1,1423191,Science Fiction


In [28]:
movie_genres = movie_genres.reset_index(drop=True)

In [29]:
movie_writers = movies_clean[["movie_id", "writers"]].explode("writers")

movie_writers = movie_writers.rename(
    columns={"writers": "writer"}
)

movie_writers = movie_writers[
    movie_writers["writer"].notna() &
    (movie_writers["writer"] != "")
]

movie_writers = movie_writers.drop_duplicates().reset_index(drop=True)

movie_writers.head()

,movie_id,writer
0,969681,Erik Sommers
1,969681,Chris McKenna
2,1423191,Zach Cregger
3,1423191,Shay Hatten
4,1101383,David Robert Mitchell


In [30]:
movie_keywords = movies_clean[["movie_id", "keywords"]].explode("keywords")

movie_keywords = movie_keywords.rename(
    columns={"keywords": "keyword"}
)

movie_keywords = movie_keywords[
    movie_keywords["keyword"].notna() &
    (movie_keywords["keyword"] != "")
]

movie_keywords = movie_keywords.drop_duplicates().reset_index(drop=True)

movie_keywords.head()

,movie_id,keyword
0,969681,new york city
1,969681,secret identity
2,969681,hero
3,969681,mutation
4,969681,villain


In [31]:
movie_cast_clean = movie_cast.copy()
movie_reviews_clean = movie_reviews.copy()

# Clean cast data
for col in ["actor", "character"]:
    movie_cast_clean[col] = (
        movie_cast_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# Clean review data
for col in ["author", "review_text"]:
    movie_reviews_clean[col] = (
        movie_reviews_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# Convert review dates
movie_reviews_clean["review_date"] = pd.to_datetime(
    movie_reviews_clean["review_date"],
    errors="coerce"
)

# Remove exact duplicate cast records
movie_cast_clean = movie_cast_clean.drop_duplicates().reset_index(drop=True)

# Each review ID should identify one review
movie_reviews_clean = movie_reviews_clean.drop_duplicates(
    subset="review_id"
).reset_index(drop=True)

In [32]:
print("Cast:", movie_cast_clean.shape)
print("Reviews:", movie_reviews_clean.shape)

movie_cast_clean.info()
movie_reviews_clean.info()

Cast: (59674, 3)
Reviews: (8900, 5)
<class 'pandas.DataFrame'>
RangeIndex: 59674 entries, 0 to 59673
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   movie_id   59674 non-null  int64 
 1   actor      59674 non-null  string
 2   character  58935 non-null  string
dtypes: int64(1), string(2)
memory usage: 2.8 MB
<class 'pandas.DataFrame'>
RangeIndex: 8900 entries, 0 to 8899
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   movie_id     8900 non-null   int64         
 1   review_id    8900 non-null   str           
 2   author       8900 non-null   string        
 3   review_date  8900 non-null   datetime64[us]
 4   review_text  8899 non-null   string        
dtypes: datetime64[us](1), int64(1), str(1), string(2)
memory usage: 5.0 MB


In [33]:
print("Total reviews:", len(movie_reviews))

print(
    "Movies with reviews:",
    movie_reviews["movie_id"].nunique()
)

print(
    "Total movies in our dataset:",
    movies["movie_id"].nunique()
)

coverage = (
    movie_reviews["movie_id"].nunique()
    / movies["movie_id"].nunique()
) * 100

print(f"Movie review coverage: {coverage:.2f}%")

Total reviews: 8900
Movies with reviews: 3839
Total movies in our dataset: 6764
Movie review coverage: 56.76%


In [35]:
movies_table = movies_clean.drop(
    columns=["genres", "writers", "keywords", "runtime"]
).copy()

movies_table = movies_table.drop_duplicates(
    subset="movie_id"
).reset_index(drop=True)

In [36]:
print("Duplicate movie IDs:", movies_table["movie_id"].duplicated().sum())

movie_ids = set(movies_table["movie_id"])

supporting_tables = {
    "movie_genres": movie_genres,
    "movie_writers": movie_writers,
    "movie_keywords": movie_keywords,
    "movie_cast": movie_cast_clean,
    "movie_reviews": movie_reviews_clean
}

for name, table in supporting_tables.items():
    unmatched = (~table["movie_id"].isin(movie_ids)).sum()

    print(
        f"{name}: {len(table)} rows, "
        f"{unmatched} rows with unmatched movie IDs"
    )

Duplicate movie IDs: 0
movie_genres: 18159 rows, 0 rows with unmatched movie IDs
movie_writers: 4635 rows, 0 rows with unmatched movie IDs
movie_keywords: 85720 rows, 0 rows with unmatched movie IDs
movie_cast: 59674 rows, 0 rows with unmatched movie IDs
movie_reviews: 8900 rows, 13 rows with unmatched movie IDs


In [37]:
movie_reviews_clean = movie_reviews_clean[
    movie_reviews_clean["movie_id"].isin(movie_ids)
].copy().reset_index(drop=True)

print("Reviews remaining:", len(movie_reviews_clean))

Reviews remaining: 8887


In [38]:
(~movie_reviews_clean["movie_id"].isin(movie_ids)).sum()

np.int64(0)

In [39]:
print("Movies:", movies_table.shape)
print("\nMissing values in movies table:")
print(movies_table.isna().sum())

print("\nDuplicate movie IDs:", movies_table["movie_id"].duplicated().sum())

print("\nDuplicate genre records:",
      movie_genres.duplicated(subset=["movie_id", "genre"]).sum())

print("Duplicate writer records:",
      movie_writers.duplicated(subset=["movie_id", "writer"]).sum())

print("Duplicate keyword records:",
      movie_keywords.duplicated(subset=["movie_id", "keyword"]).sum())

print("Duplicate cast records:",
      movie_cast_clean.duplicated(subset=["movie_id", "actor", "character"]).sum())

print("Duplicate review IDs:",
      movie_reviews_clean["review_id"].duplicated().sum())

Movies: (6764, 15)

Missing values in movies table:
movie_id                0
title                   0
movie_url               0
poster_url              6
release_date            0
overview               21
user_score            125
director               15
status                  0
original_language       0
budget               1839
revenue              1592
content_score           0
release_year            0
runtime_minutes        63
dtype: int64

Duplicate movie IDs: 0

Duplicate genre records: 0
Duplicate writer records: 0
Duplicate keyword records: 0
Duplicate cast records: 0
Duplicate review IDs: 0


In [40]:
movies_table[
    ["user_score", "content_score", "release_year", "runtime_minutes", "budget", "revenue"]
].describe()

,user_score,content_score,release_year,runtime_minutes,budget,revenue
count,6639.00000,6764.000000,6764.000000,6701.000000,4.925000e+03,5.172000e+03
mean,6.68853,99.211709,2009.447221,107.291598,4.376721e+07,1.274439e+08
std,0.83656,4.637357,15.916954,23.917527,5.114411e+07,2.146333e+08
min,1.00000,35.000000,1902.000000,2.000000,1.000000e+00,6.000000e+00
25%,6.19250,100.000000,2002.000000,94.000000,1.050000e+07,1.587426e+07
50%,6.70000,100.000000,2014.000000,105.000000,2.500000e+07,5.372882e+07
75%,7.25600,100.000000,2022.000000,119.000000,5.800000e+07,1.460959e+08
max,10.00000,100.000000,2027.000000,261.000000,6.588000e+08,2.923706e+09


In [41]:
movies_table["status"].value_counts(dropna=False)

status
Released           6723
Post Production      33
In Production         7
Planned               1
Name: count, dtype: int64

In [42]:
movies_table["original_language"].value_counts().head(15)

original_language
English               5683
Japanese               214
French                 169
Spanish; Castilian     121
Korean                 109
Italian                 66
Chinese                 65
German                  50
Cantonese               44
Hindi                   26
Tagalog                 23
Portuguese              20
Russian                 16
Tamil                   14
Danish                  14
Name: count, dtype: int64

In [43]:
print("Lowest budgets:")

movies_table.nsmallest(10, "budget")[
    ["movie_id", "title", "release_year", "budget", "revenue"]
]

Lowest budgets:


,movie_id,title,release_year,budget,revenue
5412,916405,The Quiet Girl,2022,1.0,6.0
3748,220,East of Eden,1955,2.0,5000000.0
127,1007757,Swapped,2026,7.0,7.0
6164,413337,Girl in the Box,2016,20.0,NaN
4961,1768489,In The Middle,2026,100.0,NaN
5429,620498,Fabian: Going to the Dogs,2021,178.0,NaN
4236,1592947,Sex,2026,200.0,NaN
6718,1355303,Pictureman News,2025,500.0,NaN
4504,1768217,Plan Hormiga,2026,700.0,NaN
3389,537971,Mia,2017,800.0,NaN


In [44]:
print("Lowest revenues:")

movies_table.nsmallest(10, "revenue")[
    ["movie_id", "title", "release_year", "budget", "revenue"]
]

Lowest revenues:


,movie_id,title,release_year,budget,revenue
5412,916405,The Quiet Girl,2022,1.0,6.0
127,1007757,Swapped,2026,7.0,7.0
2889,1064486,Memoir of a Snail,2024,4350000.0,7.0
1712,906126,Society of the Snow,2024,65500000.0,1281.0
6380,15849,The Mummy,1932,196000.0,1408.0
1406,11430,The Lion King 1½,2004,NaN,1465.0
5331,10234,Funny Games,1997,NaN,2506.0
4605,839,Duel,1971,450000.0,2544.0
5664,6404,Nosferatu the Vampyre,1979,1400000.0,3451.0
3572,22164,Blood and Bone,2009,3700000.0,3492.0


In [45]:
low_budget = movies_table["budget"].notna() & (movies_table["budget"] < 1000)
low_revenue = movies_table["revenue"].notna() & (movies_table["revenue"] < 1000)

print("Suspiciously low budgets:", low_budget.sum())
print("Suspiciously low revenues:", low_revenue.sum())

movies_table.loc[
    low_budget | low_revenue,
    ["movie_id", "title", "release_year", "budget", "revenue"]
]

Suspiciously low budgets: 10
Suspiciously low revenues: 3


,movie_id,title,release_year,budget,revenue
127,1007757,Swapped,2026,7.0,7.0
2889,1064486,Memoir of a Snail,2024,4350000.0,7.0
3389,537971,Mia,2017,800.0,NaN
3748,220,East of Eden,1955,2.0,5000000.0
4236,1592947,Sex,2026,200.0,NaN
4504,1768217,Plan Hormiga,2026,700.0,NaN
4961,1768489,In The Middle,2026,100.0,NaN
5412,916405,The Quiet Girl,2022,1.0,6.0
5429,620498,Fabian: Going to the Dogs,2021,178.0,NaN
6164,413337,Girl in the Box,2016,20.0,NaN


In [46]:
movies_table.loc[movies_table["budget"] < 1000, "budget"] = np.nan
movies_table.loc[movies_table["revenue"] < 1000, "revenue"] = np.nan

In [47]:
print("Budgets below $1,000:", (movies_table["budget"].dropna() < 1000).sum())
print("Revenues below $1,000:", (movies_table["revenue"].dropna() < 1000).sum())

print("Movies remaining:", len(movies_table))

Budgets below $1,000: 0
Revenues below $1,000: 0
Movies remaining: 6764


In [48]:
# Main movie table
movie_text_columns = [
    "title",
    "movie_url",
    "poster_url",
    "overview",
    "director",
    "status",
    "original_language"
]

for col in movie_text_columns:
    movies_table[col] = (
        movies_table[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


# Genres, writers and keywords
for df, col in [
    (movie_genres, "genre"),
    (movie_writers, "writer"),
    (movie_keywords, "keyword")
]:
    df[col] = df[col].astype("string").str.strip().replace("", pd.NA)


# Cast
for col in ["actor", "character"]:
    movie_cast_clean[col] = (
        movie_cast_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )


# Reviews
for col in ["author", "review_text"]:
    movie_reviews_clean[col] = (
        movie_reviews_clean[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

In [49]:
movies_table.isna().sum()

movie_id                0
title                   0
movie_url               0
poster_url              6
release_date            0
overview               21
user_score            125
director               15
status                  0
original_language       0
budget               1849
revenue              1595
content_score           0
release_year            0
runtime_minutes        63
dtype: int64

In [50]:
# Genres
movie_genres = movie_genres.dropna(subset=["movie_id", "genre"])
movie_genres = movie_genres.drop_duplicates(
    subset=["movie_id", "genre"]
).reset_index(drop=True)

# Writers
movie_writers = movie_writers.dropna(subset=["movie_id", "writer"])
movie_writers = movie_writers.drop_duplicates(
    subset=["movie_id", "writer"]
).reset_index(drop=True)

# Keywords
movie_keywords = movie_keywords.dropna(subset=["movie_id", "keyword"])
movie_keywords = movie_keywords.drop_duplicates(
    subset=["movie_id", "keyword"]
).reset_index(drop=True)

# Cast
movie_cast_clean = movie_cast_clean.dropna(
    subset=["movie_id", "actor"]
)
movie_cast_clean = movie_cast_clean.drop_duplicates(
    subset=["movie_id", "actor", "character"]
).reset_index(drop=True)

# Reviews
movie_reviews_clean = movie_reviews_clean.dropna(
    subset=["movie_id", "review_id"]
)
movie_reviews_clean = movie_reviews_clean.drop_duplicates(
    subset=["review_id"]
).reset_index(drop=True)

In [51]:
print("Movies:", movies_table.shape)
print("Genres:", movie_genres.shape)
print("Writers:", movie_writers.shape)
print("Keywords:", movie_keywords.shape)
print("Cast:", movie_cast_clean.shape)
print("Reviews:", movie_reviews_clean.shape)

Movies: (6764, 15)
Genres: (18159, 2)
Writers: (4635, 2)
Keywords: (85720, 2)
Cast: (59674, 3)
Reviews: (8887, 5)


In [52]:
movies_table.to_csv("movies_table_clean.csv", index=False, date_format="%Y-%m-%d")

movie_genres.to_csv("movie_genres_clean.csv", index=False)
movie_writers.to_csv("movie_writers_clean.csv", index=False)
movie_keywords.to_csv("movie_keywords_clean.csv", index=False)

movie_cast_clean.to_csv("movie_cast_clean.csv", index=False)
movie_reviews_clean.to_csv("movie_reviews_clean.csv", index=False)

In [55]:
from sqlalchemy import create_engine, URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="12345",
    host="localhost",
    port=3306,
    database="tmdb_movie_project"
)

engine = create_engine(connection_url)

In [56]:
from sqlalchemy import text

with engine.connect() as connection:
    print(connection.execute(text("SELECT DATABASE()")).scalar())

tmdb_movie_project


In [57]:
movies_table.to_sql(
    name="movies",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=500
)

6764

In [58]:
movie_genres.to_sql(
    name="movie_genres",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

18159

In [59]:
movie_writers.to_sql(
    name="movie_writers",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

4635

In [60]:
movie_keywords.to_sql(
    name="movie_keywords",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

85720

In [62]:
movie_cast_clean.to_sql(
    name="movie_cast",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

59674

In [63]:
movie_reviews_clean.to_sql(
    name="movie_reviews",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=500
)

8887